In [2]:
import torch

In [ ]:
class LinearRegressionScratch:
    """tensor와 자동미분만 사용한 선형회귀 모델."""

    def __init__(
        self,
        num_inputs,
        learning_rate,
        sigma=0.01,
    ):
        
        # 입력 feature의 개수
        self.num_inputs = num_inputs
        
        # SGD parameter 갱신에서 사용할 learning rate
        self.learning_rate = learning_rate
        
        # 평균 0, 표준편차 sigma인 정규분포에서 weight를 초기화한다.
        # w.shape = [num_inputs, 1]
        #
        # requires_grad=True이므로 backward()가 
        # loss에 대한 w의 gradient를 계산할 수 있다.
        self.w = torch.normal(
            mean=0.0,
            std=sigma,
            size=(num_inputs, 1),
            requires_grad=True,
        )
        
        # 출력이 하나이므로 bias도 하나다.
        #
        # b.shape = (1,)
        self.b = torch.zeros(
            1,
            requires_grad=True,
        )
        
        
    def forward(self, X):
        # X:   (batch_size, num_inputs)
        # w:   (num_inputs, 1)
        # X@w: (batch_size, 1)
        #
        # b는 broadcasting 되어 모든 sample의 출력에 더해짐.
        return X @ self.w + self.b
        
    def __call__(self, X):
        # model.forward(X) 대신 model(X)로 호출할 수 있게 한다.
        return self.forward(X)

In [4]:
model = LinearRegressionScratch(
    num_inputs=2,
    learning_rate=0.03,
    sigma=0.01,
)

print("Initial weights:")
print(model.w)
print("Weight shape:", model.w.shape)
print("Weight requires_grad:", model.w.requires_grad)

print("\nInitial bias:")
print(model.b)
print("Bias shape:", model.b.shape)
print("Bias requires_grad:", model.b.requires_grad)

Initial weights:
tensor([[ 0.0073],
        [-0.0069]], requires_grad=True)
Weight shape: torch.Size([2, 1])
Weight requires_grad: True

Initial bias:
tensor([0.], requires_grad=True)
Bias shape: torch.Size([1])
Bias requires_grad: True


In [ ]:
# sample 4개, sample마다 feature 2개
features = torch.tensor([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

predictions = model(features)

print("Features shape:", features.shape)
print("Predictions:")
print(predictions)
print("Predictions shape:", predictions.shape)

Features shape: torch.Size([4, 2])
Predictions:
tensor([[-0.0064],
        [ 0.0078],
        [-0.0054],
        [ 0.0088]], grad_fn=<AddBackward0>)
Predictions shape: torch.Size([4, 1])


In [ ]:
# 모델의 forward 계산이 실제로 Xw + b인지 확인

manual_predictions = (
    features @ model.w
    + model.b
)

print("Model predictions:")
print(predictions)

print("\nManual predictions:")
print(manual_predictions)

Model predictions:
tensor([[-0.0064],
        [ 0.0078],
        [-0.0054],
        [ 0.0088]], grad_fn=<AddBackward0>)

Manual predictions:
tensor([[-0.0064],
        [ 0.0078],
        [-0.0054],
        [ 0.0088]], grad_fn=<AddBackward0>)


In [ ]:
def squared_loss(predictions, labels):
    """미니배치의 MSE를 계산한다."""
    
    # labels를 predictions와 같은 (B, 1) shape로 맞춘다.
    labels = labels.reshape(predictions.shape)
    
    # 각 sample의 예측 오차
    errors = predictions - labels

    # sample별 손실
    per_example_loss = 0.5 * errors.pow(2)

    # 모든 sample의 손실을 평균내어 scalar로 반환한다.
    return per_example_loss.mean()

In [9]:
# 앞에서 사용한 feature 4개에 대응하는 실제 label
labels = torch.tensor([
    [1.0],
    [6.0],
    [-1.0],
    [4.0],
])

# 현재 초기 parameter로 다시 예측한다.
predictions = model(features)

# 중간 계산을 직접 확인한다.
errors = predictions - labels
per_example_loss = 0.5 * errors.pow(2)

# 정의한 함수로 미니배치 평균 loss를 계산한다.
loss = squared_loss(predictions, labels)

print("Predictions:")
print(predictions)

print("\nLabels:")
print(labels)

print("\nErrors:")
print(errors)

print("\nPer-example loss:")
print(per_example_loss)

print("\nMean loss:")
print(loss)

print("Loss shape:", loss.shape)
print("Requires gradient:", loss.requires_grad)

Predictions:
tensor([[-0.0064],
        [ 0.0078],
        [-0.0054],
        [ 0.0088]], grad_fn=<AddBackward0>)

Labels:
tensor([[ 1.],
        [ 6.],
        [-1.],
        [ 4.]])

Errors:
tensor([[-1.0064],
        [-5.9922],
        [ 0.9946],
        [-3.9912]], grad_fn=<SubBackward0>)

Per-example loss:
tensor([[ 0.5064],
        [17.9530],
        [ 0.4946],
        [ 7.9648]], grad_fn=<MulBackward0>)

Mean loss:
tensor(6.7297, grad_fn=<MeanBackward0>)
Loss shape: torch.Size([])
Requires gradient: True
